In [1]:
conda install tensorflow


Solving environment: ...working... unsuccessful initial attempt using frozen solve. Retrying with flexible solve.
Solving environment: ...working... unsuccessful attempt using repodata from current_repodata.json, retrying with next repodata source.
Solving environment: ...working... done

## Package Plan ##

  environment location: C:\ProgramData\anaconda3

  added / updated specs:
    - tensorflow


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    arrow-cpp-19.0.0           |       h33d5241_2         7.7 MB
    aws-c-auth-0.6.19          |       h2bbff1b_0          95 KB
    aws-c-cal-0.5.20           |       h2bbff1b_0          41 KB
    aws-c-common-0.8.5         |       h2bbff1b_0         198 KB
    aws-c-compression-0.2.16   |       h2bbff1b_0          21 KB
    aws-c-event-stream-0.2.15  |       hd77b12b_0          50 KB
    aws-c-http-0.6.25          |       h2bbff1b_0         181 



==> WARNING: A newer version of conda exists. <==
  current version: 23.9.0
  latest version: 25.5.1

Please update conda by running

    $ conda update -n base -c defaults conda

Or to minimize the number of packages updated during conda update use

     conda install conda=25.5.1



EnvironmentNotWritableError: The current user does not have write permissions to the target environment.
  environment location: C:\ProgramData\anaconda3




In [4]:
import tensorflow as tf
print(tf.__version__)


ModuleNotFoundError: No module named 'tensorflow'

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load dataset
df = pd.read_csv('sovereign_default_dataset_1980_2022.csv - sovereign_default_dataset_1980_2022.csv')

#  Preprocess: sort and define features
df = df.sort_values(['Country', 'Year'])
features = [
    'Debt_GDP', 'ExtDebt_GDP', 'DebtServ_XGDP', 'RealGDP_growth', 'GDP_per_capita_USD',
    'Inflation', 'fiscal_balance', 'primary_balance', 'CurrentAccount_GDP', 'Reserves_months',
    'ExchangeRate_change', 'Trade_openness', 'US_FedFundsRate', 'World_GDP_growth',
    'Oil_price_index', 'VIX', 'ICRG_political', 'ElectionYear'
]

#  Prepare sequences (5 years sliding window)
sequence_length = 5
X_sequences, y_labels = [], []

for country in df['Country'].unique():
    country_df = df[df['Country'] == country]
    for i in range(len(country_df) - sequence_length):
        seq_X = country_df.iloc[i:i+sequence_length][features].values
        target_y = country_df.iloc[i+sequence_length]['Default']
        X_sequences.append(seq_X)
        y_labels.append(target_y)

X_sequences = np.array(X_sequences)
y_labels = np.array(y_labels)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_sequences, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

# LSTM model
model = Sequential([
    LSTM(64, input_shape=(sequence_length, len(features)), return_sequences=False),
    Dropout(0.3),  # Slightly higher dropout to reduce overfitting
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping callback for efficiency
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=20,  # More epochs than your original 10
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# Extract LSTM embeddings (last hidden state output)
extractor = tf.keras.Model(inputs=model.input, outputs=model.layers[0].output)
X_train_embed = extractor.predict(X_train)
X_test_embed = extractor.predict(X_test)

# Flatten for XGBoost
X_train_embed_flat = X_train_embed.reshape((X_train_embed.shape[0], -1))
X_test_embed_flat = X_test_embed.reshape((X_test_embed.shape[0], -1))

# XGBoost on embeddings
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42,
    max_depth=4,
    learning_rate=0.05,
    n_estimators=100,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()  # Handle imbalance
)

xgb_model.fit(X_train_embed_flat, y_train)

#  Evaluate
y_pred = xgb_model.predict(X_test_embed_flat)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Hybrid model Accuracy: {accuracy:.3f}")
print(f"Hybrid model F1 Score: {f1:.3f}")


ModuleNotFoundError: No module named 'tensorflow'